In [1]:
!pip install kgbench-loader

In [2]:
!pip install torch_geometric
import torch
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from torch_geometric.utils import degree
import kgbench as kg
from torch_geometric.utils import to_undirected
# 2. Load Data (and apply your Dummy Fix if needed)
data = kg.load('dmg777k', torch=True, final=True)


loaded data dmg777k (82.79s).


In [2]:
# 1. Setup & Safety Checks
num_nodes = data.num_entities
# Ensure edge_index is within bounds
mask_valid_edges = (data.triples[:, 0] < num_nodes) & (data.triples[:, 2] < num_nodes)
# Correct edge_index construction
edge_index = torch.stack([data.triples[:, 0], data.triples[:, 2]], dim=0)[:, mask_valid_edges]
edge_type = data.triples[:, 1][mask_valid_edges]

print(f"Graph: {num_nodes} nodes, {edge_index.size(1)} edges.")


Graph: 341270 nodes, 777124 edges.


In [3]:
def get_bor_features(num_nodes, edge_index, edge_type, num_relations):
    # Create a feature matrix where x[i, r] = 1 if node i has edge type r
    # We use scatter_add to count occurrences
    x = torch.zeros(num_nodes, num_relations, device=edge_index.device)

    # Incoming edges
    # target nodes are at edge_index[1]
    target_nodes = edge_index[1]

    # Create one-hot for relations
    rel_one_hot = F.one_hot(edge_type, num_classes=num_relations).float()

    # Aggregate: Scatter sum relation vectors into target nodes
    x.scatter_add_(0, target_nodes.unsqueeze(1).expand(-1, num_relations), rel_one_hot)

    # Normalize (Log count or boolean)
    x = torch.log1p(x)
    return x

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
edge_index = edge_index.to(device)
edge_type = edge_type.to(device)

# Generate BETTER features
x_features = get_bor_features(num_nodes, edge_index, edge_type, data.num_relations).to(device)
print(f"Generated Bag-of-Relations Features: {x_features.size()}")
print("\n")
for i in range(3):
  print(x_features[i])
  print(x_features[i].sum(dim=0))
  print("\n")

Generated Bag-of-Relations Features: torch.Size([341270, 60])


tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.6931, 4.6347,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        4.6250, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000], device='cuda:0')
tensor(9.9528, device='cuda:0')


tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.6931, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.00

In [5]:
class BoR_RGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations):
        super().__init__()
        # Layer 1: BoR Features -> Hidden
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations, num_bases=30)
        # Layer 2: Hidden -> Class
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations, num_bases=30)
        self.dropout = 0.5

    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, edge_type)
        return F.log_softmax(x, dim=1)

# 4. Train
model = BoR_RGCN(data.num_relations, 64, data.num_classes, data.num_relations).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [6]:
# Prepare Labels
node_labels = torch.full((num_nodes,), -1, dtype=torch.long).to(device)
node_labels[data.training[:, 0]] = data.training[:, 1].long().to(device)
node_labels[data.withheld[:, 0]] = data.withheld[:, 1].long().to(device)
train_mask = data.training[:, 0].to(device)
test_mask = data.withheld[:, 0].to(device)

print("🚀 Starting Training with Bag-of-Relations...")
for epoch in range(101):
    model.train()
    optimizer.zero_grad()
    out = model(x_features, edge_index, edge_type)
    loss = F.nll_loss(out[train_mask], node_labels[train_mask])
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        pred = out.argmax(dim=1)
        test_acc = (pred[test_mask] == node_labels[test_mask]).sum().item() / test_mask.size(0)
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Test Acc: {test_acc:.4f}")

🚀 Starting Training with Bag-of-Relations...
Epoch 000 | Loss: 1.7269 | Test Acc: 0.1824
Epoch 010 | Loss: 1.2408 | Test Acc: 0.4968
Epoch 020 | Loss: 1.2110 | Test Acc: 0.5022
Epoch 030 | Loss: 1.1984 | Test Acc: 0.5052
Epoch 040 | Loss: 1.1959 | Test Acc: 0.5052
Epoch 050 | Loss: 1.1959 | Test Acc: 0.4988
Epoch 060 | Loss: 1.1908 | Test Acc: 0.5022
Epoch 070 | Loss: 1.1911 | Test Acc: 0.4988
Epoch 080 | Loss: 1.1900 | Test Acc: 0.5067
Epoch 090 | Loss: 1.1886 | Test Acc: 0.4988
Epoch 100 | Loss: 1.1878 | Test Acc: 0.4978


In [ ]:
for i in range(10):
    print(data.i2r[i])
    print("\n")

http://data.pdok.nl/def/pdok#asWKT-RD


http://dbpedia.org/ontology/city


http://dbpedia.org/ontology/codeNationalMonument


http://dbpedia.org/ontology/location


http://dbpedia.org/ontology/name


http://dbpedia.org/ontology/neighbourhood


http://dbpedia.org/ontology/thumbnail


http://purl.org/dc/terms/created


http://purl.org/dc/terms/creator


http://purl.org/dc/terms/description




In [7]:
images = data.get_images()

In [8]:
text_strings = data.get_strings('http://www.w3.org/2001/XMLSchema#string')
dates = data.get_strings('http://www.w3.org/2001/XMLSchema#gYear')
numbers = data.get_strings('http://www.w3.org/2001/XMLSchema#integer')

In [ ]:
all_datatypes = data.datatypes()
print("All available datatypes in DMG777K:")
for i, dtype in enumerate(all_datatypes):
    print(f"{i}: {dtype}")

All available datatypes in DMG777K:
0: iri
1: none
2: @es
3: @fy
4: @nl
5: @nl-nl
6: @pt
7: @ru
8: http://kgbench.info/dt#base64Image
9: http://www.opengis.net/ont/geosparql#wktLiteral
10: http://www.w3.org/2001/XMLSchema#anyURI
11: http://www.w3.org/2001/XMLSchema#boolean
12: http://www.w3.org/2001/XMLSchema#gYear
13: http://www.w3.org/2001/XMLSchema#nonNegativeInteger
14: http://www.w3.org/2001/XMLSchema#positiveInteger


In [9]:
geom_dtype = 'http://www.opengis.net/ont/geosparql#wktLiteral'
geom_strings = data.get_strings(geom_dtype)
print(f"Number of geographic entities: {len(geom_strings)}")

Number of geographic entities: 20837


In [10]:
year_dtype = 'http://www.w3.org/2001/XMLSchema#gYear'
years = [int(y) for y in data.get_strings(year_dtype)]
print(f"Years: {min(years)} - {max(years)}")


Years: 1100 - 8001


In [11]:
# Non-negative integers
nonneg_ints = [int(x) for x in data.get_strings('http://www.w3.org/2001/XMLSchema#nonNegativeInteger')]

# Positive integers
pos_ints = [int(x) for x in data.get_strings('http://www.w3.org/2001/XMLSchema#positiveInteger')]


In [14]:
from sklearn.feature_extraction.text import CountVectorizer
import torch

def get_text_features(data, vocab_size=1000):
    # 1. Get all text strings and their global indices
    dtype = 'http://www.w3.org/2001/XMLSchema#string' # Check exact URI in your data
    texts = data.get_strings(dtype)
    indices = data.datatype_l2g(dtype)

    # 2. Create Bag-of-Words
    # "Simple" BoW: distinct words only, limit vocab to keep it fast
    vectorizer = CountVectorizer(max_features=vocab_size, stop_words='english')
    bow_matrix = vectorizer.fit_transform(texts).toarray()

    # 3. Convert to Tensor
    return torch.tensor(bow_matrix, dtype=torch.float), indices

# Usage
feat_text, idx_text = get_text_features(data)
print(f"Text Matrix Shape: {feat_text.shape}") # (Num_Text_Nodes, 1000)

Text Matrix Shape: torch.Size([34145, 1000])


In [15]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from tqdm import tqdm

def get_image_features(data, images_list):
    # 1. Detect GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on: {device}")  # Should say 'cuda'

    # 2. Setup Model and MOVE TO GPU
    resnet = models.resnet18(pretrained=True)
    resnet = torch.nn.Sequential(*(list(resnet.children())[:-1]))
    resnet = resnet.to(device)  # <--- CRITICAL STEP 1
    resnet.eval()

    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    indices = data.datatype_l2g('http://kgbench.info/dt#base64Image')
    features = []
    batch_size = 64  # Increased batch size since GPU is faster

    with torch.no_grad():
        # Added tqdm to show progress bar
        for i in tqdm(range(0, len(images_list), batch_size)):
            batch_imgs = images_list[i:i+batch_size]

            # Preprocess on CPU (cannot be avoided easily without custom loader)
            batch_tensors = torch.stack([preprocess(img) for img in batch_imgs])

            # Move Batch to GPU
            batch_tensors = batch_tensors.to(device) # <--- CRITICAL STEP 2

            # Forward pass
            out = resnet(batch_tensors)

            # Move result back to CPU to save GPU RAM
            features.append(out.squeeze().cpu()) # <--- CRITICAL STEP 3

    return torch.cat(features), indices

# Usage (Pass your existing 'images' list here)
img_feats, img_indices = get_image_features(data, images)
print(f"Image Matrix Shape: {img_feats.shape}")


Running on: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 131MB/s]
100%|██████████| 720/720 [02:59<00:00,  4.00it/s]


Image Matrix Shape: torch.Size([46061, 512])


In [ ]:
# 1. Delete the raw images list
del images

# 2. Force garbage collection to actually reclaim the memory
import gc
gc.collect()

print("RAM cleared! Raw images deleted.")


RAM cleared! Raw images deleted.


In [21]:
import torch
import shapely.wkt
from shapely.errors import ShapelyError

def get_geo_features(data):
    # 1. Identify the correct datatype URI
    dtype = 'http://www.opengis.net/ont/geosparql#wktLiteral'

    # 2. Extract strings and indices
    geo_strings = data.get_strings(dtype)
    indices = data.datatype_l2g(dtype)

    coords = []
    fallback_count = 0

    print(f"Parsing {len(geo_strings)} geospatial strings...")

    # 3. Iterate and Parse
    for s in geo_strings:
        try:
            # FIX: Handle the specific malformed MULTIPOLYGON format from your logs
            # If it starts with MULTIPOLYGON but has no parentheses, wrap it
            if s.startswith("MULTIPOLYGON") and "(" not in s:
                clean_coords = s.replace("MULTIPOLYGON", "").strip()
                s = f"MULTIPOLYGON ((({clean_coords})))"

            # Parse with Shapely (Handles POINT, POLYGON, MULTIPOLYGON, etc.)
            geom = shapely.wkt.loads(s)

            # STANDARDISATION: Convert all geometries to a single centroid point
            # This ensures we always get [x, y] regardless of the shape complexity
            centroid = geom.centroid
            coords.append([centroid.x, centroid.y])

        except Exception as e:
            # Fallback for truly unparseable data
            # print(f"Failed on: {s[:30]}... Error: {e}") # Uncomment to debug specific failures
            coords.append([0.0, 0.0])
            fallback_count += 1

    # 4. Calculate Statistics
    total = len(geo_strings)
    if total > 0:
        fallback_pct = (fallback_count / total) * 100
        print(f"Geospatial Parsing Stats:")
        print(f"  - Total Geo Nodes: {total}")
        print(f"  - Successful: {total - fallback_count}")
        print(f"  - Fallback used: {fallback_count}")
        print(f"  - Failure Rate: {fallback_pct:.2f}%")
    else:
        print("No geospatial nodes found for this datatype.")

    return torch.tensor(coords, dtype=torch.float), indices


In [22]:
# Extract Geospatial Features
feat_geo, idx_geo = get_geo_features(data)

print(f"Geo Matrix Shape: {feat_geo.shape}")

Parsing 20837 geospatial strings...
Geospatial Parsing Stats:
  - Total Geo Nodes: 20837
  - Successful: 20837
  - Fallback used: 0
  - Failure Rate: 0.00%
Geo Matrix Shape: torch.Size([20837, 2])


In [29]:
import torch
import numpy as np
from dateutil import parser

def get_temporal_features(data):
    """
    Parses year and date literals into normalized float tensors.
    """
    # Define URIs
    dtype_year = 'http://www.w3.org/2001/XMLSchema#gYear'

    # 1. Get raw strings
    year_strings = data.get_strings(dtype_year)

    # Indices for mapping back to nodes
    year_indices = data.datatype_l2g(dtype_year)

    years = []

    # 2. Parse Years (Simple integer cast)
    for s in year_strings:
        try:
            # Handle potential negative years or edge cases if necessary
            years.append(float(s))
        except:
            years.append(np.nan) # Mark for imputation

    years = np.array(years)

    years[np.isnan(years)] = np.nanmean(years) if len(years) > 0 else 0

    # 5. Min-Max Normalization (Crucial for GNN stability)
    # Scales values to [0, 1] range to match your geospatial features
    def normalize(arr):
        if len(arr) == 0: return arr
        _min, _max = arr.min(), arr.max()
        if _max - _min == 0: return arr
        return (arr - _min) / (_max - _min)

    years = normalize(years)

    # Convert to tensors
    t_years = torch.tensor(years, dtype=torch.float).unsqueeze(1) # Shape [N, 1]

    print(f"Temporal Stats:")
    print(f"  - Parsed Years: {len(t_years)}")

    return t_years, year_indices


In [30]:
feat_year, idx_year = get_temporal_features(data)

# Create feature dictionary for your multimodal GNN
feats_dict = {
    'year': (feat_year, torch.tensor(idx_year, dtype=torch.long))
}

Temporal Stats:
  - Parsed Years: 290


In [26]:
class MultiModalEncoder(torch.nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Encoders
        self.text_encoder = nn.Linear(1000, hidden_dim)
        self.img_encoder  = nn.Linear(512, hidden_dim)
        self.geo_encoder  = nn.Linear(2, hidden_dim)
        self.year_encoder = nn.Linear(1, hidden_dim)
        self.date_encoder = nn.Linear(1, hidden_dim)

        # Base learnable embedding for all nodes (captures graph structure + fallback)
        self.node_embedding = nn.Embedding(num_nodes, hidden_dim)

    def forward(self, feats_dict, all_node_ids):
        # Base state
        x = self.node_embedding(all_node_ids)

        # Helper to add features if they exist
        def add_modality(name, encoder):
            f, idx = feats_dict.get(name, (None, None))
            if idx is not None and len(idx) > 0 and f is not None:
                # Map global node indices to local feature indices
                # (Assuming idx aligns with f rows 1-to-1)
                x[idx] = x[idx] + encoder(f.to(x.device))

        # Add all modalities
        add_modality('text', self.text_encoder)
        add_modality('image', self.img_encoder)
        add_modality('geo', self.geo_encoder)
        add_modality('year', self.year_encoder)
        add_modality('date', self.date_encoder)

        return x


In [ ]:
from torch_geometric.nn import GCNConv

class MultiModalGNN(nn.Module):
    def __init__(self, num_nodes, num_classes, hidden_dim=64):
        super().__init__()
        self.encoder = MultiModalEncoder(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, feats_dict, all_node_ids, edge_index):
        # 1. Encode Multi-Modal Features
        x = self.encoder(feats_dict, all_node_ids)

        # 2. Graph Convolutions
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

# ==========================================
# 3. Main Execution Pipeline
# ==========================================

def main():

    feat_year, idx_year, feat_date, idx_date = get_temporal_features(data)

    # Simulate Images (e.g., for 10% of nodes just to test architecture)
    # In reality, you'd parse 'http://kgbench.info/dt#base64Image'
    dummy_img_idx = torch.randperm(data.num_entities)[:int(data.num_entities*0.1)]
    feat_img, idx_img = img_feats, img_indices
    # Dictionary to pass to model
    feats_dict = {
        'geo': (feat_geo, torch.tensor(idx_geo, dtype=torch.long)),
        'year': (feat_year, torch.tensor(idx_year, dtype=torch.long)),
        'date': (feat_date, torch.tensor(idx_date, dtype=torch.long)),
        'text': (feat_text, torch.tensor(idx_text, dtype=torch.long)),
        'image': (feat_img, torch.tensor(idx_img, dtype=torch.long))
    }

    # --- B. Prepare Graph ---
    print("\n--- Constructing Graph ---")
    # kgbench uses triples (s, p, o). We need edge_index [2, E]
    # Filter for object connections (edges between entities)
    triples = data.triples
    # Mask to select object properties (not literals)
    # Note: In kgbench, num_entities is the cut-off. indices < num_entities are nodes.
    mask = (triples[:, 2] < data.num_entities)
    edges = triples[mask][:, [0, 2]].T # [2, E]
    edge_index = torch.tensor(edges, dtype=torch.long)

    # --- C. Train Model ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MultiModalGNN(num_nodes=data.num_entities,
                          num_classes=data.num_classes,
                          hidden_dim=64).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    all_node_ids = torch.arange(data.num_entities, device=device)
    edge_index = edge_index.to(device)

    y = torch.zeros(data.num_entities, dtype=torch.long)
    train_nodes = torch.tensor(data.training[:, 0], dtype=torch.long)
    train_labels = torch.tensor(data.training[:, 1], dtype=torch.long)
    y[train_nodes] = train_labels

    test_nodes = torch.tensor(data.withheld[:, 0], dtype=torch.long)
    test_labels = torch.tensor(data.withheld[:, 1], dtype=torch.long)
    y[test_nodes] = test_labels

    # 3. Now this line will work
    y = y.to(device)

    # Move features to device roughly (lazy load in model forward is safer for RAM)
    # But for small tensors, move now:
    for k in feats_dict:
        f, i = feats_dict[k]
        feats_dict[k] = (f.to(device), i.to(device))

    print("\n--- Starting Training ---")
    model.train()
    for epoch in range(51):
        optimizer.zero_grad()

        # Pass features to model
        out = model(feats_dict, all_node_ids, edge_index)

        # USE YOUR EXISTING VARIABLES HERE:
        # train_mask is your list of indices: [0, 5, 8...]
        # node_labels contains the class IDs
        loss = F.nll_loss(out[train_mask], node_labels[train_mask])

        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            model.eval()
            pred = out.argmax(dim=1)

            # Eval on test set using YOUR variables
            correct = (pred[test_mask] == node_labels[test_mask]).sum()
            acc = int(correct) / int(len(test_mask)) # len() because test_mask is a list of indices

            print(f'Epoch {epoch:03d}: Loss: {loss.item():.4f}, Test Acc: {acc:.4f}')
            model.train()

    # --- D. Final Evaluation ---
    model.eval()
    out = model(feats_dict, all_node_ids, edge_index)
    pred = out.argmax(dim=1)

    # Use node_labels instead of y
    # train_mask/test_mask are indices (LongTensor), so we use them to index

    train_correct = (pred[train_mask] == node_labels[train_mask]).sum().item()
    train_total = len(train_mask) # Since it's a list of indices
    train_acc = train_correct / train_total

    test_correct = (pred[test_mask] == node_labels[test_mask]).sum().item()
    test_total = len(test_mask)
    test_acc = test_correct / test_total

    print("\n--- Final Results ---")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy:  {test_acc:.4f}")

if __name__ == "__main__":
    main()

Temporal Stats:
  - Parsed Years: 290
  - Parsed Dates: 0

--- Constructing Graph ---


/tmp/ipython-input-2414836996.py:53: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(edges, dtype=torch.long)
/tmp/ipython-input-2414836996.py:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_nodes = torch.tensor(data.training[:, 0], dtype=torch.long)
/tmp/ipython-input-2414836996.py:67: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_labels = torch.tensor(data.training[:, 1], dtype=torch.long)
/tmp/ipython-input-2414836996.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sou


--- Starting Training ---
Epoch 000: Loss: 1.8079, Test Acc: 0.2044
Epoch 010: Loss: 1.3942, Test Acc: 0.3923
Epoch 020: Loss: 1.3274, Test Acc: 0.4123
Epoch 030: Loss: 1.2900, Test Acc: 0.4373
Epoch 040: Loss: 1.2552, Test Acc: 0.4428
Epoch 050: Loss: 1.2230, Test Acc: 0.4388

--- Final Results ---
Train Accuracy: 0.5577
Test Accuracy:  0.4608


In [3]:
from transformers import CLIPProcessor, CLIPModel
from torch_geometric.nn import RGCNConv
from tqdm.notebook import tqdm
import gc

# ==========================================
# PHASE 1: GENERATE EMBEDDINGS (CPU STAGING)
# ==========================================
print("Phase 1: Generating Embeddings...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Setup CLIP
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
hf_model = CLIPModel.from_pretrained(model_id).to(device)
hf_model.eval()

# 2. Initialize Feature Matrix on CPU (Save VRAM)
EMBED_DIM = 512
x_multimodal = torch.zeros((data.num_entities, EMBED_DIM))

# 3. Embed Images
def get_image_embeddings(images):
    embeddings = []
    with torch.no_grad():
        # Batch size 32 is safe for inference
        for i in range(0, len(images), 32):
            batch = images[i : i + 32]
            inputs = processor(images=batch, return_tensors="pt", padding=True).to(device)
            # Move result to CPU immediately
            embeddings.append(hf_model.get_image_features(**inputs).cpu())
    return torch.cat(embeddings, dim=0)

raw_images = data.get_images()
if len(raw_images) > 0:
    print(f"Embedding {len(raw_images)} images...")
    img_embeds = get_image_embeddings(raw_images)
    img_indices = data.datatype_l2g('http://kgbench.info/dt#base64Image')
    x_multimodal[img_indices] = img_embeds

# 4. Embed Text
# (Assuming text_strings is already loaded via data.get_strings)
text_strings = data.get_strings('http://www.w3.org/2001/XMLSchema#string')
raw_texts = text_strings[1] if isinstance(text_strings, tuple) else text_strings

def get_text_embeddings(texts):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), 32):
            batch = texts[i : i + 32]
            inputs = processor(text=batch, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(hf_model.get_text_features(**inputs).cpu())
    return torch.cat(embeddings, dim=0)

if len(raw_texts) > 0:
    print(f"Embedding {len(raw_texts)} text items...")
    txt_embeds = get_text_embeddings(raw_texts)
    txt_indices = data.datatype_l2g('http://www.w3.org/2001/XMLSchema#string')
    valid_count = min(len(txt_indices), len(txt_embeds))
    x_multimodal[txt_indices[:valid_count]] = txt_embeds[:valid_count]

# ==========================================
# PHASE 2: MEMORY CLEANUP (CRITICAL)
# ==========================================
print("Phase 2: Cleaning GPU Memory...")
del hf_model, processor
if 'img_embeds' in locals(): del img_embeds
if 'txt_embeds' in locals(): del txt_embeds
gc.collect()
torch.cuda.empty_cache()
print(f"GPU Free! Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")



Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Phase 1: Generating Embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Embedding 46061 images...
Embedding 34145 text items...
Phase 2: Cleaning GPU Memory...
GPU Free! Allocated: 0.01 GB


In [7]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [36]:
import torch
import os

# Define a path on your Google Drive
save_path = '/content/drive/MyDrive/dmg777k_multimodal_features.pt'

# Save the tensor
print(f"Saving features to {save_path}...")
torch.save(x_multimodal, save_path)
print("Saved successfully!")


Saving features to /content/drive/MyDrive/dmg777k_multimodal_features.pt...
Saved successfully!


In [35]:
import torch
import torch.nn.functional as F
from torch_geometric.utils import k_hop_subgraph, add_self_loops
from torch_geometric.nn import RGCNConv
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ----------------------------
# 0) GPU cleanup (recommended)
# ----------------------------
gc.collect()
torch.cuda.empty_cache()
print(f"GPU allocated now: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# ------------------------------------
# 1) Build FULL bidirectional graph
# ------------------------------------
num_nodes = data.num_entities
mask_valid = (data.triples[:, 0] < num_nodes) & (data.triples[:, 2] < num_nodes)

src = data.triples[:, 0][mask_valid]
dst = data.triples[:, 2][mask_valid]
rel = data.triples[:, 1][mask_valid].long()

edge_index_fwd = torch.stack([src, dst], dim=0)
edge_index_rev = torch.stack([dst, src], dim=0)
edge_type_fwd = rel
edge_type_rev = rel

edge_index_full = torch.cat([edge_index_fwd, edge_index_rev], dim=1).to(device)
edge_type_full  = torch.cat([edge_type_fwd, edge_type_rev], dim=0).to(device).long()

print("Full bidirectional edges:", edge_index_full.size(1))

# -------------------------------------------------------
# 2) Find a "good" subgraph: multimodal present + small
# -------------------------------------------------------
def make_subgraph(num_train, num_test, num_hops):
    train_seeds = data.training[:num_train, 0].cpu()
    test_seeds  = data.withheld[:num_test, 0].cpu()
    seed_nodes  = torch.cat([train_seeds, test_seeds]).to(device)

    subset_nodes, sub_edge_index, mapping, edge_mask = k_hop_subgraph(
        node_idx=seed_nodes,
        num_hops=num_hops,
        edge_index=edge_index_full,
        relabel_nodes=True
    )
    sub_edge_type = edge_type_full[edge_mask].long()

    # x_multimodal is on CPU typically; index with cpu ids, then move
    x_subset = x_multimodal[subset_nodes.cpu()].to(device)

    nonzero_frac = float(((x_subset.abs().sum(dim=1) > 0).float().mean()).item())
    return subset_nodes, sub_edge_index.to(device), sub_edge_type.to(device), x_subset, nonzero_frac

# Try progressively larger, but keep hops=1 for size control
candidates = [
    (1500, 500, 1),
    (2000, 800, 1),
    (3000, 1000, 1),
    (4000, 1000, 1),
    (5000, 1500, 1),
]

TARGET_MAX_NODES = 80000
TARGET_MAX_EDGES = 450000

best = None
for (nt, nv, hops) in candidates:
    subset_nodes, sub_edge_index, sub_edge_type, x_subset, nonzero_frac = make_subgraph(nt, nv, hops)
    n = x_subset.size(0)
    e = sub_edge_index.size(1)
    print(f"try train={nt}, test={nv}, hops={hops} -> nodes={n}, edges={e}, nonzero_CLIP={nonzero_frac:.3f}")

    if nonzero_frac > 0.05 and n <= TARGET_MAX_NODES and e <= TARGET_MAX_EDGES:
        best = (subset_nodes, sub_edge_index, sub_edge_type, x_subset, nonzero_frac)
        print("Selected this subgraph.")
        break

if best is None:
    raise RuntimeError("Could not find a small-enough subgraph with multimodal nodes. Reduce TARGET limits or seed sizes further.")

subset_nodes, sub_edge_index, sub_edge_type, x_subset, nonzero_frac = best
num_sub_nodes = x_subset.size(0)

# ----------------------------
# 3) Add self-loops (keep graph directed+reverse already)
# ----------------------------
self_rel = data.num_relations
sub_edge_index, _ = add_self_loops(sub_edge_index, num_nodes=num_sub_nodes)
self_edge_type = torch.full((num_sub_nodes,), self_rel, dtype=torch.long, device=device)
sub_edge_type = torch.cat([sub_edge_type, self_edge_type], dim=0)
num_relations_used = data.num_relations + 1

print(f"After self-loops: edges={sub_edge_index.size(1)} | relations={num_relations_used}")

# ----------------------------
# 4) Map labels into subgraph
# ----------------------------
node_lookup = torch.full((data.num_entities,), -1, dtype=torch.long)  # CPU
node_lookup[subset_nodes.cpu()] = torch.arange(num_sub_nodes)

train_old = data.training[:, 0]
train_in = node_lookup[train_old] >= 0
subset_train_idx = node_lookup[train_old][train_in].to(device).long()
subset_train_lbl = data.training[:, 1][train_in].long().to(device)

test_old = data.withheld[:, 0]
test_in = node_lookup[test_old] >= 0
subset_test_idx = node_lookup[test_old][test_in].to(device).long()
subset_test_lbl = data.withheld[:, 1][test_in].long().to(device)

print("Train nodes:", len(subset_train_idx), "| Test nodes:", len(subset_test_idx))

# Weighted loss
train_counts = torch.bincount(subset_train_lbl.detach().cpu(), minlength=data.num_classes).float()
class_w = (train_counts.sum() / (train_counts + 1e-6))
class_w = (class_w / class_w.mean()).to(device)

# ----------------------------
# 5) Memory-friendly model
# ----------------------------
# Store features in float16 to save VRAM, but compute in float32 inside layers
x_subset = x_subset.to(torch.float16)

class RGCN_MM(torch.nn.Module):
    def __init__(self, num_nodes, clip_dim, node_emb_dim, hidden_dim, out_dim, num_relations):
        super().__init__()
        self.node_emb  = torch.nn.Embedding(num_nodes, node_emb_dim)
        self.clip_proj = torch.nn.Linear(clip_dim, node_emb_dim)
        self.conv1 = RGCNConv(node_emb_dim, hidden_dim, num_relations, num_bases=8)
        self.conv2 = RGCNConv(hidden_dim, out_dim, num_relations, num_bases=8)

    def forward(self, x_clip, edge_index, edge_type):
        ids = torch.arange(x_clip.size(0), device=x_clip.device)
        x = self.node_emb(ids) + self.clip_proj(x_clip.float())
        x = F.relu(self.conv1(x, edge_index, edge_type))
        x = F.dropout(x, p=0.4, training=self.training)
        x = self.conv2(x, edge_index, edge_type)
        return F.log_softmax(x, dim=1)

model = RGCN_MM(
    num_nodes=num_sub_nodes,
    clip_dim=512,
    node_emb_dim=128,
    hidden_dim=128,
    out_dim=data.num_classes,
    num_relations=num_relations_used
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

print(f"GPU allocated before training: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("Training...")

best_acc, best_ep = 0.0, -1
for epoch in range(101):
    model.train()
    opt.zero_grad()
    out = model(x_subset, sub_edge_index, sub_edge_type)
    loss = F.nll_loss(out[subset_train_idx], subset_train_lbl, weight=class_w)
    loss.backward()
    opt.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            pred = out.argmax(dim=1)
            acc = (pred[subset_test_idx] == subset_test_lbl).float().mean().item()
        if acc > best_acc:
            best_acc, best_ep = acc, epoch
        print(f"Epoch {epoch:03d} | Loss {loss.item():.4f} | Test Acc {acc:.4f} | Best {best_acc:.4f}")





GPU allocated now: 1.65 GB
Full bidirectional edges: 1554248
try train=1500, test=500, hops=1 -> nodes=31034, edges=98542, nonzero_CLIP=0.070
Selected this subgraph.
After self-loops: edges=129576 | relations=61
Train nodes: 1500 | Test nodes: 500
GPU allocated before training: 1.63 GB
Training...
Epoch 000 | Loss 4.4647 | Test Acc 0.2480 | Best 0.2480
Epoch 010 | Loss 0.1132 | Test Acc 0.5760 | Best 0.5760
Epoch 020 | Loss 0.0240 | Test Acc 0.6260 | Best 0.6260
Epoch 030 | Loss 0.0091 | Test Acc 0.6760 | Best 0.6760
Epoch 040 | Loss 0.0067 | Test Acc 0.6560 | Best 0.6760
Epoch 050 | Loss 0.0061 | Test Acc 0.6560 | Best 0.6760
Epoch 060 | Loss 0.0061 | Test Acc 0.6820 | Best 0.6820
Epoch 070 | Loss 0.0075 | Test Acc 0.6800 | Best 0.6820
Epoch 080 | Loss 0.0083 | Test Acc 0.6940 | Best 0.6940
Epoch 090 | Loss 0.0075 | Test Acc 0.7000 | Best 0.7000
Epoch 100 | Loss 0.0092 | Test Acc 0.6860 | Best 0.7000
